# Qwen3.5-27B + SGLang + OpenAI Client Quickstart

この Notebook は、Docker で SGLang API サーバを起動したあとに、Python の OpenAI ライブラリから Qwen3.5-27B を使う最短手順をまとめたものです。


## 1. 前提
- Docker が使える
- NVIDIA GPU が使える
- `HF_TOKEN` が必要に応じて設定されている


In [ ]:
!docker ps --filter name=qwen35-sglang-api
!curl -s http://127.0.0.1:30000/v1/models || true


In [ ]:
!python ../scripts/generate_sample_image.py


## 2. OpenAI クライアント初期化


In [ ]:
import os
from openai import OpenAI

BASE_URL = os.environ.get('OPENAI_BASE_URL', 'http://127.0.0.1:30000/v1')
client = OpenAI(
    api_key='EMPTY',
    base_url=BASE_URL,
)
print('Using base_url =', BASE_URL)
client


## 3. テキスト推論


In [ ]:
resp = client.chat.completions.create(
    model='Qwen/Qwen3.5-27B',
    messages=[
        {'role': 'user', 'content': 'SGLangとは何かを日本語で2文で説明してください。'}
    ],
    max_tokens=64,
)
resp


## 4. 画像入力推論


In [ ]:
import base64
import mimetypes
from pathlib import Path

img = Path('../assets/sample_shapes.png')
mime = mimetypes.guess_type(img.name)[0] or 'image/png'
image_url = 'data:' + mime + ';base64,' + base64.b64encode(img.read_bytes()).decode('utf-8')

resp = client.chat.completions.create(
    model='Qwen/Qwen3.5-27B',
    messages=[
        {
            'role': 'user',
            'content': [
                {'type': 'text', 'text': 'この画像の図形の数と色を説明してください。'},
                {'type': 'image_url', 'image_url': {'url': image_url}},
            ],
        }
    ],
    max_tokens=64,
)
resp


## 5. 補足
Qwen3.5 は thinking mode が既定のため、レスポンスは `reasoning_content` 側に現れることがあります。
